---
description: 一本演示如何在 Langfuse 数据上运行 LangChain 评测的“菜谱”。
category: Evaluation
---

# 在 Langfuse 数据上运行 LangChain 评测

本指南演示如何使用基于模型的评测，自动化评估 Langfuse 中线上产出的 LLM 完成结果。示例使用 LangChain，亦可迁移到其他库；最佳选择取决于具体场景。

本指南分三步：
1. 从 Langfuse 获取线上存储的 `generations`
2. 使用 LangChain 对这些 `generations` 进行评测
3. 将结果作为 `scores` 回灌到 Langfuse


----
还未使用 Langfuse？通过采集 LLM 事件，[立即开始](https://langfuse.com/docs/get-started)。

### 环境准备

先用 pip 安装 Langfuse 与 LangChain，然后设置环境变量。

In [ ]:
%pip install langfuse langchain langchain-openai --upgrade

In [ ]:
import os

# 在项目设置页获取密钥：https://cloud.langfuse.com
os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..." 
os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..." 
os.environ["LANGFUSE_HOST"] = "https://cloud.langfuse.com" # 🇪🇺 欧盟区域
# os.environ["LANGFUSE_HOST"] = "https://us.cloud.langfuse.com" # 🇺🇸 美国区域

# 你的 OpenAI Key
os.environ["OPENAI_API_KEY"] = "sk-proj-..."

In [ ]:
os.environ['EVAL_MODEL'] = "gpt-3.5-turbo-instruct"

# LangChain 评测类型
EVAL_TYPES={
    "hallucination": True,
    "conciseness": True,
    "relevance": True,
    "coherence": True,
    "harmfulness": True,
    "maliciousness": True,
    "helpfulness": True,
    "controversiality": True,
    "misogyny": True,
    "criminality": True,
    "insensitivity": True
}

初始化 Langfuse Python SDK，更多信息见[此处](https://langfuse.com/docs/sdk/python#1-installation)。

In [ ]:
from langfuse import get_client
 
langfuse = get_client()
 
# 验证连接
if langfuse.auth_check():
    print("Langfuse 客户端已通过认证，准备就绪！")
else:
    print("认证失败。请检查凭据与主机配置。")

Langfuse client is authenticated and ready!


### 拉取数据

根据 `name` 从 Langfuse 载入所有 `generations`，此处示例为 `OpenAI`。在 Langfuse 中，`name` 用于标识应用内不同类型的生成。将其替换为你需要评测的名称。

关于在写入 LLM Generation 时如何设置 `name`，参见[文档](https://langfuse.com/docs/sdk/python#generation)。

In [24]:
def fetch_all_pages(name=None, user_id = None, limit=50):
    page = 1
    all_data = []

    while True:
        response = langfuse.api.trace.list(name=name, limit=limit, user_id=user_id, page=page)
        if not response.data:
            break

        all_data.extend(response.data)
        page += 1

    return all_data

In [35]:
generations = fetch_all_pages(user_id='user_123')

In [40]:
generations[0].id

'adb5ba6beab14984ab89006ee09e9cd6'

### 定义评测函数

本节基于 `EVAL_TYPES` 定义 LangChain 评测器；其中“幻觉”（hallucination）需要单独函数。关于 LangChain 评测的更多信息见[此处](https://python.langchain.com/docs/guides/evaluation/string/criteria_eval_chain)。

In [37]:
from langchain.evaluation import load_evaluator
from langchain_openai import OpenAI
from langchain.evaluation.criteria import LabeledCriteriaEvalChain

def get_evaluator_for_key(key: str):
  llm = OpenAI(temperature=0, model=os.environ.get('EVAL_MODEL'))
  return load_evaluator("criteria", criteria=key, llm=llm)

def get_hallucination_eval():
  criteria = {
    "hallucination": (
      "Does this submission contain information"
      " not present in the input or reference?"
    ),
  }
  llm = OpenAI(temperature=0, model=os.environ.get('EVAL_MODEL'))

  return LabeledCriteriaEvalChain.from_llm(
      llm=llm,
      criteria=criteria,
  )

### 执行评测

下面将对上面载入的每个 `Generation` 执行评测。每个得分将通过 [`langfuse.score()`](https://langfuse.com/docs/scores) 写回 Langfuse。


In [ ]:
def execute_eval_and_score():

  for generation in generations:
    criteria = [key for key, value in EVAL_TYPES.items() if value and key != "hallucination"]

    for criterion in criteria:
      eval_result = get_evaluator_for_key(criterion).evaluate_strings(
          prediction=generation.output,
          input=generation.input,
      )
      print(eval_result)

      langfuse.create_score(name=criterion, trace_id=generation.id, observation_id=generation.id, value=eval_result["score"], comment=eval_result['reasoning'])

execute_eval_and_score()


In [ ]:
# 幻觉（hallucination）

def eval_hallucination():

  chain = get_hallucination_eval()

  for generation in generations:
    eval_result = chain.evaluate_strings(
      prediction=generation.output,
      input=generation.input,
      reference=generation.input
    )
    print(eval_result)
    if eval_result is not None and eval_result["score"] is not None and eval_result["reasoning"] is not None:
      langfuse.create_score(name='hallucination', trace_id=generation.id, observation_id=generation.id, value=eval_result["score"], comment=eval_result['reasoning'])


In [29]:
if EVAL_TYPES.get("hallucination") == True:
  eval_hallucination()

In [ ]:
# SDK 为异步实现，请确保等待所有请求完成
langfuse.flush()

### 在 Langfuse 中查看分数

在 Langfuse 界面中，你可以按 `Scores` 过滤 Traces，并查看每条的详细信息。你也可以通过 Langfuse Analytics 分析新提示词版本或应用发布对这些分数的影响。

![Trace 示例](https://langfuse.com/images/docs/trace-conciseness-score.jpg)
_包含简洁性（conciseness）分数的示例 Trace_


## 联系我们

想要为你的线上数据设计更贴合场景的打分方式？欢迎加入我们的 [Discord](https://langfuse.com/discord) 讨论你的用例！